# 10. End-to-End Real-Time Scoring & SAR Auto-Generation

**Author:** UPADRASTHA VIDHATH  
**Domain:** Anti-Money Laundering (AML) & Financial Intelligence  
**System:** Intelligent Transaction Monitoring & Behavioral Analytics  

---

### Executive Summary
Production inference simulation: scoring live incoming financial transactions, risk tiering, and auto-generating Suspicious Activity Reports (SAR).


## 1. Load Production Pipeline


In [1]:
import joblib
import pandas as pd
import numpy as np

model = joblib.load('final_aml_detection_model.joblib')
print("Loaded production AML model.")


## 2. Real-Time Transaction Ingestion Simulation


In [1]:
new_transactions = pd.DataFrame([
    {
        "step": 240, "log_amount": np.log1p(9850.0), "oldbalanceOrg": 10000.0, "newbalanceOrig": 150.0,
        "oldbalanceDest": 200.0, "newbalanceDest": 10050.0, "isFlaggedFraud": 0, "customerRiskScore": 88.5,
        "countryRiskIndex": 3, "transactionVelocity": 8, "isStructuring": 1, "errorBalanceOrig": 0.0,
        "errorBalanceDest": 0.0, "amount_to_oldbalance_ratio": 0.985, "type_CASH_OUT": 0, "type_DEBIT": 0,
        "type_PAYMENT": 0, "type_TRANSFER": 1
    },
    {
        "step": 240, "log_amount": np.log1p(45.0), "oldbalanceOrg": 5200.0, "newbalanceOrig": 5155.0,
        "oldbalanceDest": 8000.0, "newbalanceDest": 8045.0, "isFlaggedFraud": 0, "customerRiskScore": 12.0,
        "countryRiskIndex": 1, "transactionVelocity": 1, "isStructuring": 0, "errorBalanceOrig": 0.0,
        "errorBalanceDest": 0.0, "amount_to_oldbalance_ratio": 0.008, "type_CASH_OUT": 0, "type_DEBIT": 0,
        "type_PAYMENT": 1, "type_TRANSFER": 0
    }
])

preds = model.predict(new_transactions)
probs = model.predict_proba(new_transactions)[:, 1]

for i, (pred, prob) in enumerate(zip(preds, probs)):
    risk_tier = "CRITICAL" if prob > 0.8 else "HIGH" if prob > 0.5 else "MEDIUM" if prob > 0.2 else "LOW"
    status = "🚨 MONEY LAUNDERING SUSPICION (FILE SAR)" if pred == 1 else "✅ NORMAL TRANSACTION"
    print(f"Transaction #{i+1}: Probability={prob*100:.2f}% | Tier={risk_tier} | Status: {status}")


## 3. Automated Suspicious Activity Report (SAR) Generation


In [1]:
def generate_sar_report(tx_id, orig, dest, amount, risk_score, indicators):
    sar = f'''
    ================================================================================
    SUSPICIOUS ACTIVITY REPORT (SAR) - FINANCIAL CRIMES ENFORCEMENT NETWORK
    ================================================================================
    Report Reference ID : SAR-2026-AML-{tx_id}
    Originator Account  : {orig}
    Beneficiary Account : {dest}
    Transaction Amount  : ${amount:,.2f}
    Risk Assessment Score: {risk_score:.2f} / 100.0 (CRITICAL TIER)
    Triggered Red Flags : {', '.join(indicators)}
    Compliance Action   : Flagged for mandatory FinCEN Section 314(a) audit.
    ================================================================================
    '''
    return sar

print(generate_sar_report("99824", "C10482910", "C99823412", 9850.00, 96.8, ["Structuring evasion (<$10k)", "High velocity (8 tx/24h)", "FATF Greylist Jurisdiction"]))
